# Revisão reproduzível do treinamento

## tl;dr

Este notebook reexecuta as verificações principais usadas no relatório de 21/08/2026: status dos runs, métricas de treino/validação, métricas de teste, previsões e telemetria. Ele é somente-leitura e deve ser executado a partir da raiz do repositório.

## Context & Methods

A unidade de análise é um run do benchmark. O run concluído usado para resultados finais é `batch-032/mnist`; o run ativo no snapshot é `batch-064/mnist`. Os arquivos fonte são os JSON/CSV gravados pelo próprio benchmark.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter

ROOT = Path('outputs/controlled-augmentation2-mac-m4')
COMPLETED = ROOT / 'batch/batch-032/mnist/runs/mnist__unit_interval__all_raw__seed-42'
ACTIVE = ROOT / 'batch/batch-064/mnist/runs/mnist__unit_interval__all_raw__seed-42'


## Data

### 1. Status dos runs


In [ ]:
statuses = []
for path in ROOT.glob('**/runs/*/status.json'):
    with path.open() as f:
        data = json.load(f)
    statuses.append(data.get('status'))
print({'total_status_files': len(statuses), 'status_counts': dict(Counter(statuses))})


### 2. Métricas de treino e validação


In [ ]:
def read_metrics(run_root):
    with (run_root / 'checkpoints/epoch_metrics.csv').open(newline='') as f:
        return list(csv.DictReader(f))

completed_history = read_metrics(COMPLETED)
active_history = read_metrics(ACTIVE)
for label, history in [('completed_batch32', completed_history), ('active_batch64', active_history)]:
    best = max(history, key=lambda row: float(row['val_macro_f1']))
    last = history[-1]
    print(label, {'epochs': len(history), 'best_epoch': int(best['epoch']), 'best_val_macro_f1': float(best['val_macro_f1']), 'last_epoch': int(last['epoch']), 'last_val_macro_f1': float(last['val_macro_f1']), 'last_val_accuracy': float(last['val_accuracy'])})


### 3. Teste e previsões


In [ ]:
with (COMPLETED / 'artifacts/test_metrics.json').open() as f:
    test_metrics = json.load(f)['classification']
with (COMPLETED / 'artifacts/predictions.csv').open(newline='') as f:
    predictions = list(csv.DictReader(f))
errors = [row for row in predictions if row['true_label'] != row['predicted_label']]
error_pairs = Counter((row['true_label'], row['predicted_label']) for row in errors)
print({'test_accuracy': test_metrics['accuracy'], 'test_balanced_accuracy': test_metrics['balanced_accuracy'], 'test_macro_f1': test_metrics['macro_f1'], 'test_samples': len(predictions), 'prediction_errors': len(errors), 'top_error_pairs': error_pairs.most_common(5)})


### 4. Telemetria


In [ ]:
for label, run_root in [('completed_batch32', COMPLETED), ('active_batch64', ACTIVE)]:
    with (run_root / 'telemetry/samples.csv').open(newline='') as f:
        samples = list(csv.DictReader(f))
    print(label, {'rows': len(samples), 'fields': len(samples[0]) if samples else 0, 'first_timestamp': samples[0]['timestamp'] if samples else None, 'last_timestamp': samples[-1]['timestamp'] if samples else None})


## Results

Os resultados executados devem confirmar que o batch 32 concluiu 100 épocas e possui avaliação final, enquanto o batch 64 possui somente métricas intermediárias até a última linha persistida.

## Takeaways

- O run concluído tem desempenho de teste próximo ao melhor desempenho de validação.
- As previsões estão salvas linha a linha e os principais erros podem ser auditados pela matriz de confusão.
- A matriz completa continua pendente; não usar os comparativos globais atuais como resultado final até serem regenerados.